# Minimal GR00T-N1.7 fine-tune on `stack_without_ft_tact_v4`

Standalone notebook, no `launch_finetune.py` subprocess, no `training_cfgs` YAML,
no sbatch wrapper. Data loading goes through `robotdataset.LeRobotVideoDataset`
(the package's own LeRobot-v2.1-with-external-video loader); the model and its
processor are GR00T's own tested classes, used directly rather than through the
`Gr00tN1d7Pipeline`/`DatasetFactory`/HF-`Trainer` machinery.

Grounded facts from earlier characterization (`outputs/data_stats.csv`,
`gr00t_contract.md`):
- 51 episodes, 12,244 frames, 15 fps, single task "pick the lego block."
- EEF action (dims 0:6) is **RELATIVE** (state delta); gripper (dim 6) is
  **ABSOLUTE** continuous.
- `meta/stats.json` + `meta/relative_stats.json` already generated via
  `gr00t/data/stats.py` against `configs/embodiment_config.py`
  (`EmbodimentTag.NEW_EMBODIMENT`, `ActionFormat.XYZ_ROTVEC` for the 6-dim pose).

Run this inside `$HPCWORK/gr00t-ft/venvs/gr00t` (`module load FFmpeg/7.1.2` first).


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import torch
import wandb

REPO_ROOT = Path.home() / "finetune-groot"
sys.path.insert(0, str(REPO_ROOT / "vendor" / "Isaac-GR00T"))
sys.path.insert(0, str(REPO_ROOT / "configs"))  # for embodiment_config.py

DATASET_PATH = Path(os.environ["HPCWORK"]) / "gr00t-ft" / "data" / "stack_without_ft_tact_v4"
CHECKPOINT_DIR = Path(os.environ["HPCWORK"]) / "gr00t-ft" / "checkpoints" / "notebook_run"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


## Register the embodiment config

Importing `embodiment_config.py` mutates GR00T's global `MODALITY_CONFIGS`
registry as a side effect (same as `load_modality_config` in
`launch_finetune.py`) -- this must happen before building the model config or
the processor.


In [ ]:
import embodiment_config  # noqa: F401  -- registers EmbodimentTag.NEW_EMBODIMENT

from gr00t.configs.data.embodiment_configs import MODALITY_CONFIGS
from gr00t.data.embodiment_tags import EmbodimentTag

EMBODIMENT_TAG = EmbodimentTag.NEW_EMBODIMENT
modality_configs = MODALITY_CONFIGS[EMBODIMENT_TAG.value]
print(list(modality_configs.keys()))


## Build the model

`Gr00tN1d7Config` defaults + the four tuning flags validated in the
calibration run (projector + diffusion head trainable, backbone frozen).
Loading in bf16 matches the calibration run's working configuration.


In [ ]:
from gr00t.configs.model.gr00t_n1d7 import Gr00tN1d7Config
from gr00t.model.gr00t_n1d7.gr00t_n1d7 import Gr00tN1d7
from gr00t.model.gr00t_n1d7.processing_gr00t_n1d7 import Gr00tN1d7Processor

BASE_MODEL_PATH = "nvidia/GR00T-N1.7-3B"

model_config = Gr00tN1d7Config(
    tune_llm=False,
    tune_visual=False,
    tune_projector=True,
    tune_diffusion_model=True,
)

model, loading_info = Gr00tN1d7.from_pretrained(
    BASE_MODEL_PATH,
    tune_llm=model_config.tune_llm,
    tune_visual=model_config.tune_visual,
    tune_projector=model_config.tune_projector,
    tune_diffusion_model=model_config.tune_diffusion_model,
    tune_vlln=model_config.tune_vlln,
    state_dropout_prob=model_config.state_dropout_prob,
    backbone_trainable_params_fp32=model_config.backbone_trainable_params_fp32,
    load_bf16=True,
    output_loading_info=True,
    trust_remote_code=True,
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f}%)")


## Build the processor and load normalization statistics

`meta/stats.json` holds per-column stats keyed by raw parquet column name
(`observation.state.cartesian`, `action`, ...); `meta/relative_stats.json`
holds the RELATIVE-action stats keyed by the action's `state_key` (`eef`).
`processor.set_statistics` expects `{modality: {joint_key: {stat: values}}}`
plus a top-level `relative_action` entry -- this reproduces exactly what
`LeRobotEpisodeLoader.get_dataset_statistics()` assembles, without pulling in
the loader itself.


In [ ]:
import json

processor = Gr00tN1d7Processor(
    modality_configs={EMBODIMENT_TAG.value: modality_configs},  # parse_modality_configs expects {tag: {modality: cfg}}
    use_percentiles=model_config.use_percentiles,
    statistics=None,
    embodiment_id_mapping=None,
    image_crop_size=model_config.image_crop_size,
    image_target_size=model_config.image_target_size,
    random_rotation_angle=model_config.random_rotation_angle,
    color_jitter_params=model_config.color_jitter_params,
    model_name=model_config.model_name,
    model_type=model_config.backbone_model_type,
    formalize_language=model_config.formalize_language,
    max_state_dim=model_config.max_state_dim,
    max_action_dim=model_config.max_action_dim,
    apply_sincos_state_encoding=model_config.apply_sincos_state_encoding,
    max_action_horizon=model_config.action_horizon,
    use_albumentations=model_config.use_albumentations_transforms,
    extra_augmentation_config=model_config.extra_augmentation_config,
    shortest_image_edge=model_config.shortest_image_edge,
    crop_fraction=model_config.crop_fraction,
    letter_box_transform=model_config.letter_box_transform,
    use_relative_action=model_config.use_relative_action,
    exclude_state=model_config.exclude_state,
    state_dropout_prob=model_config.state_dropout_prob,
    use_mean_std=model_config.use_mean_std,
)


def load_dataset_statistics(dataset_path: Path, modality_configs: dict) -> dict:
    """Reproduce LeRobotEpisodeLoader.get_dataset_statistics() without the loader."""
    meta_dir = dataset_path / "meta"
    with open(meta_dir / "stats.json") as f:
        stats = json.load(f)
    rel_path = meta_dir / "relative_stats.json"
    if rel_path.exists():
        with open(rel_path) as f:
            relative_stats = json.load(f)
        relative_stats.pop("__fingerprints__", None)
        stats["relative_action"] = relative_stats
    with open(meta_dir / "modality.json") as f:
        modality_meta = json.load(f)

    mapping = {"state": "observation.state", "action": "action"}
    out: dict = {}
    for modality in ("state", "action"):
        out[modality] = {}
        for joint_key in modality_configs[modality].modality_keys:
            meta = modality_meta[modality][joint_key]
            stats_key = meta.get("original_key") or mapping[modality]
            start, end = meta["start"], meta["end"]
            out[modality][joint_key] = {
                stat_type: values[start:end]
                for stat_type, values in stats[stats_key].items()
            }
    if "relative_action" in stats:
        out["relative_action"] = stats["relative_action"]
    return out


dataset_statistics = load_dataset_statistics(DATASET_PATH, modality_configs)
processor.set_statistics({EMBODIMENT_TAG.value: dataset_statistics})
print("statistics loaded for:", list(processor.statistics.keys()))


## Load data via `robotdataset.LeRobotVideoDataset`

46 episodes for training, the other 5 held out for evaluation (fixed split,
recorded here rather than randomized per run). `delta_timestamps` mirrors
`embodiment_config.py`: video/state anchor-only, action a 16-step /15fps
window (~1.07s) to match the registered `action.delta_indices`.


In [ ]:
from robotdataset import LeRobotVideoDataset

ALL_EPISODES = list(range(51))
VAL_EPISODES = [46, 47, 48, 49, 50]
TRAIN_EPISODES = [e for e in ALL_EPISODES if e not in VAL_EPISODES]

ACTION_HORIZON = 16          # matches embodiment_config.py action.delta_indices
FPS = 15.0
BATCH_SIZE = 8

action_dt = [i / FPS for i in range(ACTION_HORIZON)]

train_ds = LeRobotVideoDataset(
    local_path=str(DATASET_PATH),
    episodes=TRAIN_EPISODES,
    batch_size=BATCH_SIZE,
    delta_timestamps={"action": action_dt},
    control_frequency=FPS,
)
print("train episodes:", train_ds.num_episodes, "total steps:", len(train_ds))

val_ds = LeRobotVideoDataset(
    local_path=str(DATASET_PATH),
    episodes=VAL_EPISODES,
    batch_size=BATCH_SIZE,
    delta_timestamps={"action": action_dt},
    control_frequency=FPS,
)
print("val episodes:", val_ds.num_episodes, "total steps:", len(val_ds))


## Adapter: robotdataset batch -> `VLAStepData` -> processor -> model inputs

`robotdataset` returns the raw 7-dim `action` column as one block (matching
the dataset's own parquet schema); it is sliced here into `eef` (dims 0:6)
and `gripper` (dim 6:7) to match `embodiment_config.py`'s two `ActionConfig`
entries. Video/state default to anchor-only (`T=1`), action carries the
16-step horizon.


In [ ]:
from gr00t.data.types import MessageType, VLAStepData


def batch_to_vla_steps(batch) -> list[VLAStepData]:
    """One VLAStepData per sample in a robotdataset batch (anchor-only video/state)."""
    b = batch.batch_size[0]
    steps = []
    action = batch["action"].numpy()  # (B, ACTION_HORIZON, 7)
    primary = batch["observation/images/primary"].numpy()  # (B, 1, C, H, W)
    wrist = batch["observation/images/wrist"].numpy()
    cartesian = batch["observation/state/cartesian"].numpy()  # (B, 1, 6)
    gripper_s = batch["observation/state/gripper"].numpy()
    joints = batch["observation/state/joints"].numpy()
    target = batch["observation/state/target"].numpy()
    task = batch.get("task", None)

    for i in range(b):
        images = {
            "primary": [np.transpose(primary[i, 0], (1, 2, 0))],  # CHW -> HWC
            "wrist": [np.transpose(wrist[i, 0], (1, 2, 0))],
        }
        states = {
            "cartesian": cartesian[i],
            "gripper": gripper_s[i].reshape(1, -1) if gripper_s[i].ndim == 1 else gripper_s[i],
            "joints": joints[i],
            "target": target[i],
        }
        actions = {
            "eef": action[i, :, :6],
            "gripper": action[i, :, 6:7],
        }
        text = str(task[i]) if task is not None else "pick the lego block."
        steps.append(VLAStepData(
            images=images, states=states, actions=actions,
            text=text, embodiment=EMBODIMENT_TAG,
        ))
    return steps


def steps_to_model_batch(steps: list[VLAStepData]):
    features = [
        processor([{"type": MessageType.EPISODE_STEP.value, "content": step}])
        for step in steps
    ]
    return processor.collator(features)


# Smoke-test the adapter on one batch before training.
_sample = train_ds.sample()
_steps = batch_to_vla_steps(_sample)
_model_batch = steps_to_model_batch(_steps)
print("model batch keys:", list(_model_batch.keys()) if hasattr(_model_batch, "keys") else type(_model_batch))


## Training loop

Minimal: AdamW over trainable params only, cosine schedule, wandb logging,
periodic checkpoint. No HF `Trainer` -- every step is explicit.


In [ ]:
MAX_STEPS = 200          # calibration-scale; raise for the full run
LEARNING_RATE = 1e-4
WARMUP_STEPS = 10
SAVE_EVERY = 100
LOG_EVERY = 10

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: min(1.0, (step + 1) / WARMUP_STEPS)
    * max(0.0, (MAX_STEPS - step) / max(1, MAX_STEPS - WARMUP_STEPS)),
)

wandb.init(project="finetune-gr00t-stack-tact", name="notebook_run", config={
    "max_steps": MAX_STEPS, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
    "action_horizon": ACTION_HORIZON, "trainable_params": trainable_params,
})

model.train()
processor.train()
losses = []
for step in range(MAX_STEPS):
    batch = train_ds.sample()
    steps = batch_to_vla_steps(batch)
    model_batch = steps_to_model_batch(steps)
    model_batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in model_batch.items()}

    out = model(**model_batch)
    loss = out.loss if hasattr(out, "loss") else out["loss"]

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable, 1.0)
    optimizer.step()
    scheduler.step()

    losses.append(loss.item())
    if step % LOG_EVERY == 0 or step == MAX_STEPS - 1:
        wandb.log({"train/loss": loss.item(), "train/lr": scheduler.get_last_lr()[0]}, step=step)
        print(f"step {step:4d}  loss {loss.item():.4f}")
    if (step + 1) % SAVE_EVERY == 0:
        ckpt_dir = CHECKPOINT_DIR / f"step_{step+1}"
        model.save_pretrained(ckpt_dir)
        processor.save_pretrained(ckpt_dir)
        print("saved checkpoint:", ckpt_dir)

wandb.finish()


## Loss curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(losses)
ax.set_xlabel("step")
ax.set_ylabel("training loss")
ax.set_title("Minimal notebook training loop")
fig.tight_layout()
fig.savefig("notebook_run_loss.png", dpi=150)
plt.show()
